# Super-Resolution Example

This notebook demonstrates how to use the PyFibreBundle package to produce an enhaned resolution image from multiple shifted images acquired through a fibre bundle.

## Overview
- Load sample images (stack of shifted images and calibration/background image)
- Reconstruct a single image using triangular linear interpolation
- Reconstruct and enhanced image using multiple shifted images
  
## Getting Started

Click Run -> Run All Cells

## Setup: Import Libraries

In [ ]:
import os, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import context

from pybundle import PyBundle

## Load Images

We load:
1. A stack of shifted images
2. A background image for calibration/normalisation.


In [ ]:
dataFolder = Path('../test/data/super_res/data')
backFile = Path('../test/data/super_res/background.tif')

shift = None
nImages = None

# Find all image files in the folder
files = [f.path for f in os.scandir(dataFolder)]

if nImages is None:
    nImages = len(files)

print(f'Found {len(files)} images in stack folder')
print(f'Using nImages = {nImages}')

In [ ]:
# Load stack into a 3D array: (height, width, nImages)
img = np.array(Image.open(files[0]))
imSize = np.shape(img)
imgs = np.zeros((imSize[0], imSize[1], nImages), dtype='uint8')

for idx, fName in enumerate(files[:nImages]):
    img = Image.open(fName)
    imgs[:, :, idx] = np.array(img)

calibImg = np.array(Image.open(backFile))

print(f'Stack shape: {imgs.shape}')
print(f'Calibration image shape: {calibImg.shape}')

## Visualise Input

Show the first frame in the stack and the calibration image.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

axes[0].imshow(imgs[:, :, 0], cmap='gray')
axes[0].set_title('First Input Frame')
axes[0].axis('off')

axes[1].imshow(calibImg, cmap='gray')
axes[1].set_title('Background/Calibration Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Single-Image Reconstruction

This section performs standard reconstruction on a single frame for comparison.

In [ ]:
pyb_single = PyBundle(
    coreMethod=PyBundle.TRILIN,
    gridSize=800,
    coreSize=3,
    calibImage=calibImg,
    normaliseImage=calibImg
)

pyb_single.calibrate()
reconSingle = pyb_single.process(imgs[:, :, 0])



In [ ]:
plt.figure(dpi=150)
plt.imshow(reconSingle, cmap='gray')
plt.title('Single Image Reconstruction')
plt.axis('off')
plt.show()

## Super-Resolution Reconstruction

Enable super-resolution and provide the shifted calibration image stack.

In [ ]:
pyb_sr = PyBundle(
    coreMethod=PyBundle.TRILIN,
    gridSize=800,
    coreSize=3,
    calibImage=calibImg,
    normaliseImage=calibImg,
    superRes=True,
    srCalibImages=imgs
)

t1 = time.perf_counter()
pyb_sr.calibrate_sr()
t2 = time.perf_counter()

print(f'Super-res one-time calibration took: {round(1000 * (t2 - t1))} ms')

t1 = time.perf_counter()
reconSR = pyb_sr.process(imgs)
t2 = time.perf_counter()

print(f'Super-res reconstruction took: {round(1000 * (t2 - t1))} ms')


In [ ]:
plt.figure(dpi=150)
plt.imshow(reconSR, cmap='gray')
plt.title('Resolution-Enhanced Image')
plt.axis('off')
plt.show()

## Comparison

Visual side-by-side comparison of single-image and super-res outputs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

axes[0].imshow(reconSingle, cmap='gray')
axes[0].set_title('Single Image')
axes[0].axis('off')

axes[1].imshow(reconSR, cmap='gray')
axes[1].set_title('Resolution-Enhanced Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()